In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]

In [0]:
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("HealthcarePipeline").getOrCreate()
df = spark.createDataFrame(data, columns)
df.show()


+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
df2 = df.withColumn(
    "total_bill",
    col("consultation_fee") + (col("tests_count") * 500)
).withColumn(
    "patient_type",
    when(col("total_bill") >= 6000, "High")
    .when(col("total_bill") >= 4000, "Medium")
    .otherwise("Low")
)
df2.show()

+--------+------------+---------+-----------+----------------+-----------+----------+------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|patient_type|
+--------+------------+---------+-----------+----------------+-----------+----------+------------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|      Medium|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|      Medium|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|         Low|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|        High|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|        High|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|      Medium|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5500|      Medium|
|     108|

In [0]:
high_value_df = df2.filter(col("patient_type") == "High")
high_value_df.show()

+--------+------------+---------+----------+----------------+-----------+----------+------------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|total_bill|patient_type|
+--------+------------+---------+----------+----------------+-----------+----------+------------+
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      6000|        High|
|     105|Vikram Singh|  Chennai| Neurology|            7000|          1|      7500|        High|
+--------+------------+---------+----------+----------------+-----------+----------+------------+



In [0]:
agg_df = df2.groupBy("department").agg(
    count("*").alias("total_patients"),
    sum("total_bill").alias("total_revenue"),
    avg("total_bill").alias("avg_bill")
)
agg_df.show()

+-----------+--------------+-------------+-----------------+
| department|total_patients|total_revenue|         avg_bill|
+-----------+--------------+-------------+-----------------+
| Cardiology|             3|        17000|5666.666666666667|
|Orthopedics|             2|         8500|           4250.0|
|Dermatology|             2|         4500|           2250.0|
|  Neurology|             1|         7500|           7500.0|
+-----------+--------------+-------------+-----------------+



In [0]:
sorted_df = agg_df.orderBy(col("total_revenue").desc())
sorted_df.show()

+-----------+--------------+-------------+-----------------+
| department|total_patients|total_revenue|         avg_bill|
+-----------+--------------+-------------+-----------------+
| Cardiology|             3|        17000|5666.666666666667|
|Orthopedics|             2|         8500|           4250.0|
|  Neurology|             1|         7500|           7500.0|
|Dermatology|             2|         4500|           2250.0|
+-----------+--------------+-------------+-----------------+



Part 2

In [0]:
df2.createOrReplaceTempView("patient_visits")

In [0]:
%sql
SELECT * FROM patient_visits WHERE department = 'Cardiology';


visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Medium
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Medium


In [0]:
%sql
SELECT city,SUM(total_bill) AS total_revenue
FROM patient_visits
GROUP BY city
ORDER BY total_revenue DESC;

city,total_revenue
Bangalore,8500
Chennai,7500
Hyderabad,5500
Ahmedabad,5500
Kolkata,4500
Delhi,4000
Mumbai,2000


In [0]:
%sql
SELECT * FROM patient_visits
ORDER BY total_bill DESC
LIMIT 5;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_type
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Medium
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Medium
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Medium


In [0]:
%sql
SELECT department,COUNT(*) AS patient_count
FROM patient_visits
GROUP BY department
ORDER BY patient_count DESC;

department,patient_count
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


Delta Lake


In [0]:
%sql
CREATE TABLE patient_delta
USING DELTA
AS
SELECT * FROM patient_visits;

num_affected_rows,num_inserted_rows


In [0]:
%sql
INSERT INTO patient_delta VALUES
(109,"sureshkrishna","Chennai","Cardiology",2000,2,6000,"High"),
(110,"ramesh","Chennai","Neurology",3000,1,5500,"High");

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
UPDATE patient_delta
SET consultation_fee = consultation_fee + 500
WHERE department = 'Cardiology';

num_affected_rows
4


In [0]:
%sql
DELETE FROM patient_delta
WHERE patient_type = 'Low';

num_affected_rows
2


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW updates AS
SELECT * FROM VALUES
(101,"Arjun Reddy","Hyderabad","Cardiology",5500,2,6500,"High"),
(111,"Neha Sharma","Delhi","Dermatology",2000,1,2500,"Low")
AS updates(
visit_id,
patient_name,
city,
department,
consultation_fee,
tests_count,
total_bill,
patient_type
);

MERGE INTO patient_delta AS target
USING updates AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
UPDATE SET
target.patient_name = source.patient_name,
target.city = source.city,
target.department = source.department,
target.consultation_fee = source.consultation_fee,
target.tests_count = source.tests_count,
target.total_bill = source.total_bill,
target.patient_type = source.patient_type

WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
DESCRIBE HISTORY patient_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-05-04T05:17:58.000Z,142952842377702,azuser6411_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(286006913283086),082d005f-95fe-4b65-8410-3971470dbf0b,0504-035749-hxynjl0k-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7315, p25FileSize -> 2849, numDeletionVectorsRemoved -> 1, minFileSize -> 2849, numAddedFiles -> 1, maxFileSize -> 2849, p75FileSize -> 2849, p50FileSize -> 2849, numAddedBytes -> 2849)",null,Databricks-Runtime/18.1.x-photon-scala2.13
6,2026-05-04T05:17:56.000Z,142952842377702,azuser6411_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#16509L = cast(visit_id#16493 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(286006913283086),082d005f-95fe-4b65-8410-3971470dbf0b,0504-035749-hxynjl0k-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4521, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3874, materializeSourceTimeMs -> 396, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1569, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1863)",null,Databricks-Runtime/18.1.x-photon-scala2.13
5,2026-05-04T05:17:31.000Z,142952842377702,azuser6411_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(286006913283086),7a59b613-8218-46c5-8bc4-71eef33a66cb,0504-035749-hxynjl0k-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2867, p25FileSize -> 2794, numDeletionVectorsRemoved -> 1, minFileSize -> 2794, numAddedFiles -> 1, maxFileSize -> 2794, p75FileSize -> 2794, p50FileSize -> 2794, numAddedBytes -> 2794)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-04T05:17:30.000Z,142952842377702,azuser6411_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(patient_type#16135 = Low)""])",null,List(286006913283086),7a59b613-8218-46c5-8bc4-71eef33a66cb,0504-035749-hxynjl0k-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1649, numDeletionVectorsUpdated -> 0, numDeletedRows -> 2, scanTimeMs -> 1293, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 355)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-05-04T05:17:15.000Z,142952842377702,azuser6411_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(286006913283086),6b12fe82-e03f-4c47-bede-222b993adcf0,0504-035749-hxynjl0k-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7831, p25FileSize -> 2867, numDeletionVectorsRemoved -> 2, minFileSize -> 2867, numAddedFiles -> 1, maxFileSize -> 2867, p75FileSize -> 2867, p50FileSize -> 2867, numAddedBytes -> 2867)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-05-04T05:17:14.000Z,142952842377702,azuser6411_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(department#15524 = Cardiology)""])",null,List(286006913283086),6b12fe82-e03f-4c47-

In [0]:
%sql
SELECT *
FROM patient_delta VERSION AS OF 0;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Medium
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,4000,Medium
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2000,Low
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Medium
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Medium
108,Meera Iyer,Bangalore,Dermatology,1500,2,2500,Low


In [0]:
%sql VACUUM patient_delta RETAIN 168 HOURS DRY RUN;

path


In [0]:
df2.write.mode("overwrite").saveAsTable("patient_parquet")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE patient_delta
USING DELTA
AS SELECT * FROM patient_parquet
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("SELECT * FROM patient_delta").show()
spark.sql("DESCRIBE DETAIL patient_delta").show()

+--------+------------+---------+-----------+----------------+-----------+----------+------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|patient_type|
+--------+------------+---------+-----------+----------------+-----------+----------+------------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|      Medium|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|      Medium|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|         Low|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|        High|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|        High|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|      Medium|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5500|      Medium|
|     108|

In [0]:
%sql
CREATE TABLE patient_target
USING DELTA
AS
SELECT * FROM patient_visits;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_updates AS
SELECT * FROM VALUES
(101,"Arjun Reddy","Hyderabad","Cardiology",6000,2,7000,"High"), 
(112,"Pooja Shah","Mumbai","Neurology",8000,1,8500,"High") 
AS daily_updates(
visit_id,
patient_name,
city,
department,
consultation_fee,
tests_count,
total_bill,
patient_type
);

In [0]:
%sql
MERGE INTO patient_target AS target
USING daily_updates AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
UPDATE SET
target.patient_name = source.patient_name,
target.city = source.city,
target.department = source.department,
target.consultation_fee = source.consultation_fee,
target.tests_count = source.tests_count,
target.total_bill = source.total_bill,
target.patient_type = source.patient_type

WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1
